# ✦ MagicBrush Pro - High Performance GPU Backend
This notebook provides the GPU-accelerated Stable Diffusion Inpainting and LPIPS scoring server.

### 🚀 Setup Instructions
1. **Runtime Type**: Ensure you are on a **T4 GPU** (Runtime -> Change runtime type).
2. **Ngrok Token**: Get your free token from [ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken) and paste it in the **Run Server** cell at the bottom.
3. **Run All**: Press `Ctrl + F9` to run all cells.
4. **Connect**: Copy the `ngrok.io` URL printed at the bottom and paste it into your local `frontend/app.js` file.

In [24]:
!pip install fastapi python-multipart uvicorn[standard] torch torchvision diffusers transformers accelerate lpips Pillow numpy opencv-python-headless pyngrok nest-asyncio

In [25]:
utils_code = """import base64
import io
import numpy as np
from PIL import Image

def decode_base64_image(base64_str: str) -> Image.Image:
    if "," in base64_str:
        base64_str = base64_str.split(",")[1]
    image_data = base64.b64decode(base64_str)
    return Image.open(io.BytesIO(image_data)).convert('RGB')

def encode_image_base64(image: Image.Image) -> str:
    buffered = io.BytesIO()
    image.save(buffered, format="PNG")
    img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")
    return f"data:image/png;base64,{img_str}"

def preprocess_image(image: Image.Image):
    return image.resize((512, 512), Image.Resampling.LANCZOS).convert("RGB")
"""

with open('utils.py', 'w') as f:
    f.write(utils_code)
print('utils.py created.')

utils.py created.


In [26]:
scoring_code = """import torch
import lpips
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torchvision.transforms as transforms

class Scorer:
    def __init__(self, device="cuda" if torch.cuda.is_available() else "cpu"):
        self.device = device
        print(f"Loading CLIP & LPIPS models on {device}...")
        self.clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(self.device)
        self.clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
        self.lpips_model = lpips.LPIPS(net='vgg').to(self.device)
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])

    def score(self, original_img: Image.Image, gen_img: Image.Image, prompt: str):
        with torch.no_grad():
            # CLIP Match
            inputs = self.clip_processor(text=[prompt], images=gen_img, return_tensors="pt", padding=True).to(self.device)
            clip_score = self.clip_model(**inputs).logits_per_image.item()

            # LPIPS Distance
            img_a = self.transform(original_img).unsqueeze(0).to(self.device)
            img_b = self.transform(gen_img).unsqueeze(0).to(self.device)
            lpips_score = self.lpips_model(img_a, img_b).item()

        scaled_clip = clip_score / 40.0
        final_score = (0.7 * scaled_clip) - (0.3 * lpips_score)
        return {"clip_score": round(clip_score, 2), "lpips_score": round(lpips_score, 4), "final_score": round(final_score, 2)}

_scorer = None
def get_scorer():
    global _scorer
    if _scorer is None: _scorer = Scorer()
    return _scorer
"""

with open('scoring.py', 'w') as f:
    f.write(scoring_code)
print('scoring.py created.')

scoring.py created.


In [27]:
gallery_code = """import json, os, time
from typing import List, Dict, Any

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
GALLERY_FILE = os.path.join(BASE_DIR, "gallery_history.json")

class GalleryManager:
    def __init__(self, storage_path: str = GALLERY_FILE):
        self.storage_path = storage_path
        if not os.path.exists(self.storage_path):
            with open(self.storage_path, "w") as f: json.dump([], f)

    def add_entry(self, entry: Dict[str, Any]):
        entry["id"] = str(int(time.time() * 1000))
        entry["timestamp"] = time.strftime("%Y-%m-%d %H:%M:%S")
        history = self.get_all_entries()
        history.insert(0, entry)
        history = history[:15] # Keep size small for UI stability
        with open(self.storage_path, "w") as f: json.dump(history, f)
        return entry["id"]

    def get_all_entries(self) -> List[Dict[str, Any]]:
        try:
            if not os.path.exists(self.storage_path): return []
            with open(self.storage_path, "r") as f: return json.load(f)
        except: return []

    def delete_entry(self, entry_id: str) -> bool:
        history = self.get_all_entries()
        new_h = [e for e in history if e.get("id") != entry_id]
        if len(new_h) < len(history):
            with open(self.storage_path, "w") as f: json.dump(new_h, f)
            return True
        return False

    def get_entry(self, entry_id: str):
        for e in self.get_all_entries():
            if e.get("id") == entry_id: return e
        return None

_mgr = None
def get_gallery_manager():
    global _mgr
    if _mgr is None: _mgr = GalleryManager()
    return _mgr
"""

with open('gallery_manager.py', 'w') as f:
    f.write(gallery_code)
print('gallery_manager.py created.')

gallery_manager.py created.


In [28]:
pipeline_code = """import torch
from diffusers import StableDiffusionInpaintPipeline
from PIL import Image

class InpaintPipeline:
    def __init__(self, device="cuda" if torch.cuda.is_available() else "cpu"):
        self.device = device
        dtype = torch.float16 if device == "cuda" else torch.float32
        self.pipe = StableDiffusionInpaintPipeline.from_pretrained(
            "runwayml/stable-diffusion-inpainting",
            torch_dtype=dtype, safety_checker=None
        ).to(self.device)
        if device == "cuda": self.pipe.enable_attention_slicing()

    def generate(self, image: Image.Image, mask: Image.Image, prompt: str, num_samples: int = 4):
        image = image.resize((512, 512)).convert("RGB")
        mask = mask.resize((512, 512)).convert("L")
        print(f"Batched Gen: {num_samples} samples for '{prompt}'")
        return self.pipe(
            prompt=prompt, image=image, mask_image=mask,
            num_inference_steps=30, guidance_scale=7.5, num_images_per_prompt=num_samples
        ).images

_pipe = None
def get_pipeline():
    global _pipe
    if _pipe is None: _pipe = InpaintPipeline()
    return _pipe
"""

with open('model_pipeline.py', 'w') as f:
    f.write(pipeline_code)
print('model_pipeline.py created (Optimized Batched Gen).')

model_pipeline.py created (Optimized Batched Gen).


In [29]:
main_code = """from fastapi import FastAPI, HTTPException, Form
from fastapi.middleware.cors import CORSMiddleware
from utils import decode_base64_image, encode_image_base64, preprocess_image
from scoring import get_scorer
from model_pipeline import get_pipeline
from gallery_manager import get_gallery_manager
import base64, io, traceback, json, numpy as np
from PIL import Image, ImageFilter

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

def decode_mask(mask_b64: str) -> np.ndarray:
    if "," in mask_b64: mask_b64 = mask_b64.split(",")[1]
    img = Image.open(io.BytesIO(base64.b64decode(mask_b64))).convert("RGBA").resize((512,512))
    alpha = np.array(img)[:, :, 3].astype(np.float32) / 255.0
    alpha = (alpha > 0.1).astype(np.float32)
    pil_a = Image.fromarray((alpha * 255).astype(np.uint8), mode="L").filter(ImageFilter.MaxFilter(5))
    return np.array(pil_a).astype(np.float32)[:, :, np.newaxis] / 255.0

def hard_composite(original, generated, mask_alpha):
    orig = np.array(original.resize((512,512))).astype(np.float32)
    gen = np.array(generated.resize((512,512))).astype(np.float32)
    return Image.fromarray((gen * mask_alpha + orig * (1.0 - mask_alpha)).clip(0,255).astype(np.uint8))

@app.on_event("startup")
async def startup():
    print("Warming up models..."); get_pipeline(); get_scorer()
    mgr = get_gallery_manager()
    try:
        h = mgr.get_all_entries()
        if len(h) > 15:
            with open(mgr.storage_path, "w") as f: json.dump(h[:15], f)
    except: pass

@app.post("/generate")
async def generate(image: str=Form(...), prompt: str=Form(...), mask: str=Form(None)):
    try:
        orig_img = decode_base64_image(image)
        proc_img = preprocess_image(orig_img)
        if mask and len(mask) > 100:
            mask_alpha = decode_mask(mask)
            mask_pil = Image.fromarray((mask_alpha[:,:,0]*255).astype(np.uint8), "L")
        else:
            mask_alpha = np.ones((512,512,1), dtype=np.float32)
            mask_pil = Image.new("L", (512,512), 255)

        gens = get_pipeline().generate(proc_img, mask_pil, prompt)
        comps = [hard_composite(proc_img, g, mask_alpha) for g in gens]

        scorer = get_scorer()
        results = []
        for g in comps:
            results.append({"image": encode_image_base64(g), "scores": scorer.score(proc_img, g, prompt)})
        results.sort(key=lambda x: x["scores"]["final_score"], reverse=True)

        get_gallery_manager().add_entry({"original_image": image, "mask_image": mask, "prompt": prompt, "outputs": results, "final_image": results[0]["image"]})
        return {"samples": results}
    except Exception as e:
        print(traceback.format_exc()); raise HTTPException(500, str(e))

@app.get("/gallery")
async def gallery(): return get_gallery_manager().get_all_entries()

@app.delete("/gallery/{id}")
async def delete_g(id: str): return {"status": "success" if get_gallery_manager().delete_entry(id) else "failed"}

@app.get("/remix/{id}")
async def remix(id: str):
    item = get_gallery_manager().get_entry(id)
    if not item: raise HTTPException(404)
    return item
"""

with open('main.py', 'w') as f:
    f.write(main_code)
print('main.py created.')

main.py created.


In [ ]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn
import main

NGROK_TOKEN = "3AqWkiRTYQvG3sxXBdMch1xcr4c_4W9pABNZUW35EG2NRtSF2"

ngrok.set_auth_token(NGROK_TOKEN)

nest_asyncio.apply()

ngrok.kill()

tunnel = ngrok.connect(8000, "http")

print("\n" + "="*60)
print("✦ MAGICBRUSH PRO GPU BACKEND IS ONLINE ✦")
print("="*60)
print(f"\n-> COPY THIS URL: {tunnel.public_url}")
print("\n-> PASTE IT IN: frontend/app.js (line 2)")
print("="*60 + "\n")

config = uvicorn.Config(main.app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)

await server.serve()


✦ MAGICBRUSH PRO GPU BACKEND IS ONLINE ✦

-> COPY THIS URL: https://centaurial-caridad-unusurious.ngrok-free.dev

-> PASTE IT IN: frontend/app.js (line 2)



INFO:     Started server process [628]
INFO:     Waiting for application startup.


Warming up models...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

...

INFO:     202.88.252.190:0 - "POST /generate HTTP/1.1" 200 OK
Batched Gen: 4 samples for 'remove the microwave and stove and put a refrigerator in it's place'


  0%|          | 0/30 [00:00<?, ?it/s]

INFO:     202.88.252.190:0 - "POST /generate HTTP/1.1" 200 OK
Batched Gen: 4 samples for 'add hot air balloon'


  0%|          | 0/30 [00:00<?, ?it/s]

INFO:     202.88.252.190:0 - "POST /generate HTTP/1.1" 200 OK
